# 1. Introduction

This notebook presents a beginner-friendly deep learning project for customer churn prediction using an Artificial Neural Network (ANN).

The objective is to predict whether a telecom customer is likely to churn (Yes) or stay (No) based on customer profile and service details.

The complete workflow is intentionally simple, structured, and easy to explain in viva.

# 2. Import Libraries

We import only the required libraries for data handling, visualization, preprocessing, model building, and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

sns.set_style('whitegrid')

# 3. Load Dataset

Now we load the Telco Customer Churn dataset and inspect a few records.

In [ ]:
file_path = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(file_path)

print('Dataset shape:', df.shape)
df.head()

# 4. Exploratory Data Analysis (basic)

A quick EDA helps us understand data types, class distribution, and basic numeric behavior.

In [ ]:
print('Column data types:')
print(df.dtypes)

print('\nChurn value counts:')
print(df['Churn'].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Churn Distribution')
plt.show()

df[['tenure', 'MonthlyCharges']].describe()

# 5. Data Preprocessing

In this step we clean the dataset:
- handle missing values
- convert target labels to numeric
- remove non-predictive identifier columns

In [ ]:
data = df.copy()

# TotalCharges may contain blank strings, so convert safely to numeric
data['TotalCharges'] = data['TotalCharges'].replace(' ', np.nan)
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')

# Fill missing TotalCharges with median
data['TotalCharges'] = data['TotalCharges'].fillna(data['TotalCharges'].median())

# Convert target labels to 0/1
data['Churn'] = data['Churn'].map({'Yes': 1, 'No': 0})

# customerID is just an identifier
if 'customerID' in data.columns:
    data = data.drop(columns=['customerID'])

print('Missing values after preprocessing:', data.isnull().sum().sum())
data.head()

# 6. Feature Engineering

We create a few simple and explainable features:
- tenure_group (new/mid/long-term customer)
- charges_per_tenure (monthly spending behavior)

Then we encode categorical variables using one-hot encoding.

In [ ]:
data['tenure_group'] = pd.cut(
    data['tenure'],
    bins=[-1, 12, 48, 72],
    labels=['new', 'mid', 'long']
)

data['charges_per_tenure'] = data['MonthlyCharges'] / (data['tenure'] + 1)

X = data.drop(columns=['Churn'])
y = data['Churn']

X_encoded = pd.get_dummies(X, drop_first=True)

print('Encoded feature shape:', X_encoded.shape)
X_encoded.head()

# 7. Train-Test Split

We split data into training and testing sets.
After splitting, numerical features are scaled for stable ANN training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

feature_columns = X_train.columns

num_cols_to_scale = ['tenure', 'MonthlyCharges', 'TotalCharges', 'charges_per_tenure']
num_cols_to_scale = [col for col in num_cols_to_scale if col in X_train.columns]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols_to_scale] = scaler.fit_transform(X_train[num_cols_to_scale])
X_test_scaled[num_cols_to_scale] = scaler.transform(X_test[num_cols_to_scale])

print('Train shape:', X_train_scaled.shape)
print('Test shape:', X_test_scaled.shape)

# 8. Build ANN Model

We use a simple ANN architecture:
- Hidden Layer 1: 32 neurons, ReLU
- Hidden Layer 2: 16 neurons, ReLU
- Output Layer: 1 neuron, Sigmoid

In [ ]:
model = Sequential()
model.add(Dense(32, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# 9. Train Model

Now we train the ANN model and store training history for plots.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    verbose=1
)

# 10. Model Evaluation

We evaluate the model using:
- Accuracy
- Confusion matrix
- Classification report

In [ ]:
y_pred_prob = model.predict(X_test_scaled).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix:')
print(cm)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, digits=4))

# 11. Visualization (accuracy & loss graphs)

These graphs show learning behavior across epochs.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# 12. Simple Prediction Function

The function below keeps explainability simple:
- churn probability
- risk category (Low / Medium / High)
- 2-3 short reasons based on tenure and monthly charges

In [ ]:
def preprocess_single_input(input_dict):
    single_df = pd.DataFrame([input_dict]).copy()

    single_df['TotalCharges'] = single_df['TotalCharges'].replace(' ', np.nan)
    single_df['TotalCharges'] = pd.to_numeric(single_df['TotalCharges'], errors='coerce')
    single_df['TotalCharges'] = single_df['TotalCharges'].fillna(data['TotalCharges'].median())

    if 'customerID' in single_df.columns:
        single_df = single_df.drop(columns=['customerID'])

    single_df['tenure_group'] = pd.cut(
        single_df['tenure'],
        bins=[-1, 12, 48, 72],
        labels=['new', 'mid', 'long']
    )
    single_df['charges_per_tenure'] = single_df['MonthlyCharges'] / (single_df['tenure'] + 1)

    single_encoded = pd.get_dummies(single_df, drop_first=True)
    single_encoded = single_encoded.reindex(columns=feature_columns, fill_value=0)

    single_scaled = single_encoded.copy()
    single_scaled[num_cols_to_scale] = scaler.transform(single_scaled[num_cols_to_scale])

    return single_scaled

def explain_prediction(input_dict, probability):
    reasons = []

    if input_dict['tenure'] <= 12:
        reasons.append('Customer tenure is low, so churn risk can increase.')
    elif input_dict['tenure'] >= 48:
        reasons.append('Customer tenure is high, which often improves retention.')

    if input_dict['MonthlyCharges'] >= 80:
        reasons.append('Monthly charges are high, which may increase churn risk.')
    elif input_dict['MonthlyCharges'] <= 40:
        reasons.append('Monthly charges are lower, which may support retention.')

    if probability >= 0.65:
        reasons.append('Overall model output indicates high churn chance.')
    elif probability >= 0.35:
        reasons.append('Overall model output indicates medium churn chance.')
    else:
        reasons.append('Overall model output indicates low churn chance.')

    return reasons[:3]

def predict_churn_with_explanation(input_dict):
    prepared = preprocess_single_input(input_dict)
    probability = float(model.predict(prepared, verbose=0)[0][0])

    if probability < 0.35:
        risk = 'Low'
    elif probability < 0.65:
        risk = 'Medium'
    else:
        risk = 'High'

    reasons = explain_prediction(input_dict, probability)

    print('Prediction Summary')
    print('-' * 45)
    print(f'Churn Probability : {probability:.2%}')
    print(f'Risk Category     : {risk} Risk')
    print('Reasons:')
    for i, reason in enumerate(reasons, start=1):
        print(f'{i}. {reason}')

    return probability, risk, reasons

In [ ]:
# Sample prediction 1
sample_1 = df.iloc[0].to_dict()
_ = predict_churn_with_explanation(sample_1)

print('\n')

# Sample prediction 2
sample_2 = df.iloc[3].to_dict()
_ = predict_churn_with_explanation(sample_2)

# 13. Conclusion

In this notebook, we built a clean ANN-based churn prediction workflow with clear preprocessing, simple feature engineering, training, evaluation, and basic explainability.

Key takeaways:
- The ANN model predicts churn with standard classification metrics.
- Tenure and monthly charges give practical churn insights.
- The final prediction function is short, readable, and viva-friendly.